# Task 1

Your solution:

Prompt: I am studying the paper "Centroid Terms as Text Representatives" and would like to explore the concept of Text-representing Centroids.

Could you explain whether and how it is possible to implement the concept of TRCs using static word embeddings?

Please provide a detailed response covering the following aspects:

Implementation Method: How exactly can TRCs be constructed or represented using static word vectors?

Necessary Conditions: What specific conditions or prerequisites (data-wise or architectural) are required for this implementation?

Advantages and Disadvantages: What are the pros and cons of using static embeddings for TRCs compared to other methods like dynamic/contextual embeddings?

Application Fields: Which real-world applications or NLP tasks are most suitable for this specific approach?

Please ensure the explanation is technical yet clear.

Which version of ChatGPT you used: GPT 5.4

Yes. TRCs can be implemented with static word embeddings, but not in the exact graph-theoretic form of the paper unless you translate the original idea into embedding space.

In the paper, a centroid term is not the arithmetic average of word vectors. It is a term whose average shortest-path distance to the document’s words in a global co-occurrence graph is minimal. Importantly, the centroid term does not need to appear in the document itself, and document similarity can then be defined through distances between centroid terms. 

Centroid_Terms_as_Text_Represen…

 

Centroid_Terms_as_Text_Represen…

So the main question is: can we preserve that “text-representing centroid term” idea with static embeddings? The answer is yes, in two practical ways.

1) How to implement TRCs with static word vectors
A. Nearest-term-to-document-centroid formulation

This is the simplest embedding-based analogue.

Let a document 
𝐷
=
{
𝑤
1
,
…
,
𝑤
𝑛
}
D={w
1
	​

,…,w
n
	​

}.
Assume each word has a static embedding 
𝑒
(
𝑤
𝑖
)
∈
𝑅
𝑑
e(w
i
	​

)∈R
d
.

First compute a weighted document representation:

𝑣
𝐷
=
∑
𝑖
=
1
𝑛
𝛼
𝑖
𝑒
(
𝑤
𝑖
)
∑
𝑖
=
1
𝑛
𝛼
𝑖
v
D
	​

=
∑
i=1
n
	​

α
i
	​

∑
i=1
n
	​

α
i
	​

e(w
i
	​

)
	​


where 
𝛼
𝑖
α
i
	​

 could be:

TF

TF-IDF

POS-based weights

keyphrase/attention-like heuristic weights

Then search over a vocabulary 
𝑉
V for a word 
𝑡
t whose embedding is closest to 
𝑣
𝐷
v
D
	​

:

𝑡
∗
=
arg
⁡
min
⁡
𝑡
∈
𝑉
′
  
d
i
s
t
(
𝑒
(
𝑡
)
,
𝑣
𝐷
)
t
∗
=arg
t∈V
′
min
	​

dist(e(t),v
D
	​

)

where 
𝑉
′
V
′
 is the candidate centroid vocabulary.

That 
𝑡
∗
t
∗
 becomes the TRC.

This preserves an important property of the original paper: the representative term may be outside the document, because you are searching over a global vocabulary rather than only the document tokens. The original method also explicitly allows centroid terms that are not present in the document. 

Centroid_Terms_as_Text_Represen…

B. Medoid-like embedding TRC

A closer analogue to the paper is to pick the vocabulary term minimizing average distance to the document words:

𝑡
∗
=
arg
⁡
min
⁡
𝑡
∈
𝑉
′
1
𝑛
∑
𝑖
=
1
𝑛
𝛽
𝑖
⋅
d
i
s
t
(
𝑒
(
𝑡
)
,
𝑒
(
𝑤
𝑖
)
)
t
∗
=arg
t∈V
′
min
	​

n
1
	​

i=1
∑
n
	​

β
i
	​

⋅dist(e(t),e(w
i
	​

))

This is more faithful to the paper’s definition, where the centroid term minimizes average distance to document words in the co-occurrence graph. In the original, the document-to-term distance is defined as the average shortest-path distance from a candidate term to the document’s reachable words, and the centroid term is the term with minimum such distance. 

Centroid_Terms_as_Text_Represen…

If you replace graph shortest-path distance with cosine distance in embedding space, you get an embedding-based TRC:

d
i
s
t
(
𝑥
,
𝑦
)
=
1
−
cos
⁡
(
𝑥
,
𝑦
)
dist(x,y)=1−cos(x,y)

This version is usually better than plain vector averaging when your goal is specifically to output an interpretable term, not just a document vector.

C. Hybrid graph + static embedding TRC

This is often the strongest practical design.

The paper’s TRC depends on a global co-occurrence graph built from a corpus, with edge weights reflecting significance such as Dice coefficient, and shortest-path distances over that graph. 

Centroid_Terms_as_Text_Represen…

You can keep that structure but use static embeddings in either of two roles:

Edge enhancement: create graph edges not only from co-occurrence but also from embedding similarity.

Candidate restriction / smoothing: use embeddings to expand sparse vocabulary neighborhoods or back off when the graph is disconnected.

A hybrid distance might be:

𝑑
hybrid
(
𝑎
,
𝑏
)
=
𝜆
 
𝑑
graph
(
𝑎
,
𝑏
)
+
(
1
−
𝜆
)
 
(
1
−
cos
⁡
(
𝑒
(
𝑎
)
,
𝑒
(
𝑏
)
)
)
d
hybrid
	​

(a,b)=λd
graph
	​

(a,b)+(1−λ)(1−cos(e(a),e(b)))

Then define the centroid term as the candidate minimizing average hybrid distance to the document words.

This is conceptually closest to the paper while overcoming sparsity problems.

2) What exact pipeline would this look like?

A practical static-embedding TRC pipeline could be:

Step 1: preprocess text

The paper removes stopwords and keeps mainly nouns, proper nouns, and names in base form before building the graph. 

Centroid_Terms_as_Text_Represen…


For an embedding version, that is still a good idea because TRCs are supposed to be topical representatives, not function words.

Typical preprocessing:

tokenization

lemmatization

stopword removal

retain content-bearing POS

optional named entity extraction

Step 2: choose static embeddings

Examples:

word2vec

GloVe

fastText

fastText is often best if morphology or OOV handling matters.

Step 3: define candidate vocabulary 
𝑉
′
V
′

Do not search over every token in the raw vocabulary unless you want noisy outputs. Better choices:

nouns/proper nouns only

terms above a corpus frequency threshold

domain lexicon

concept inventory from your corpus

Step 4: compute document weights

For each word 
𝑤
𝑖
w
i
	​

, define 
𝛼
𝑖
α
i
	​

 or 
𝛽
𝑖
β
i
	​

:

TF-IDF

TextRank / keyword score

noun bonus

position weight for titles/headlines

Step 5: compute TRC

Choose one of:

nearest to weighted document centroid

medoid-like minimum average distance

hybrid graph + embedding formulation

Step 6: document similarity

Once each document gets a TRC 
𝑡
𝐷
t
D
	​

, you can define similarity like the paper does via distance between centroid terms. The paper defines document distance as the distance between the two centroid terms, then converts it to a similarity score. 

Centroid_Terms_as_Text_Represen…

In embedding form:

s
i
m
(
𝐷
1
,
𝐷
2
)
=
cos
⁡
(
𝑒
(
𝑡
𝐷
1
)
,
𝑒
(
𝑡
𝐷
2
)
)
sim(D
1
	​

,D
2
	​

)=cos(e(t
D
1
	​

	​

),e(t
D
2
	​

	​

))

or

s
i
m
(
𝐷
1
,
𝐷
2
)
=
1
1
+
d
i
s
t
(
𝑒
(
𝑡
𝐷
1
)
,
𝑒
(
𝑡
𝐷
2
)
)
sim(D
1
	​

,D
2
	​

)=
1+dist(e(t
D
1
	​

	​

),e(t
D
2
	​

	​

))
1
	​


You can also combine TRC similarity with full document-vector similarity.

3) Necessary conditions and prerequisites

There are several important conditions.

A. A stable static embedding space

You need pretrained static embeddings whose geometry reflects the semantic relations relevant to your corpus. If the domain is specialized, generic embeddings may fail badly.

Examples:

biomedical corpora need biomedical embeddings

legal corpora need legal-domain vectors

social media may need slang-aware vectors

Without this, the “best representative term” may be semantically off.

B. A good candidate vocabulary

This is critical. In the original paper, centroid terms come from a reference co-occurrence graph built from a preferably large corpus, and the graph can be built from any corpus or from the same document set being analyzed. 

Centroid_Terms_as_Text_Represen…


In embedding TRCs, the analogue is a well-chosen global candidate vocabulary.

If your vocabulary is too broad:

you get generic words like “thing,” “system,” “process”

If too narrow:

you miss good abstractions

So you often need:

frequency filtering

POS filtering

domain-specific concept lists

optional hypernym inventories if you want more abstract TRCs

C. Content-word-focused document representation

Static embeddings are poor at representing syntax and discourse. TRCs are much more stable when built from:

nouns

noun phrases

named entities

domain terms

This aligns with the paper’s own preprocessing choice. 

Centroid_Terms_as_Text_Represen…

D. Optional corpus-level structure

If you want a faithful reproduction of the paper, you need more than embeddings. You need a reference structure analogous to the paper’s co-occurrence graph, because the original approach relies on corpus-derived relational distances rather than only local document content. 

Centroid_Terms_as_Text_Represen…

So there are really two implementation levels:

Minimal implementation: only static embeddings + candidate vocabulary

Stronger implementation: static embeddings + corpus graph/statistics

E. Sufficient lexical coverage

Static embeddings require that most important terms are covered. OOV words, rare names, and new terminology are a common weakness. fastText helps because subword modeling can generate vectors for unseen forms.

F. A distance metric aligned with the task

Common choices:

cosine distance

Euclidean distance

Mahalanobis distance in whitened space

learned metric on top of static embeddings

Cosine is usually the default.

4) What are the advantages of static embeddings for TRCs?
1. Simplicity

Implementation is straightforward and computationally cheap.

A medoid-style TRC over static vectors is much easier than running a transformer over every document-candidate pair.

2. Interpretable output

You get a single term or a small set of terms, which is exactly the spirit of the paper. The paper explicitly frames centroid terms as document representatives and topical centers. 

Centroid_Terms_as_Text_Represen…

3. Works even when the representative term is absent from the document

This is one of the most interesting properties of the original method, and the static-embedding version can preserve it. 

Centroid_Terms_as_Text_Represen…

4. Low resource requirements

Useful when:

GPU is unavailable

documents are short

latency matters

models must run on-device

5. Domain control

Static embeddings can be trained on a specific corpus, which sometimes gives more stable domain terminology than large general-purpose contextual models.

6. Better than pure bag-of-words for lexical variation

The paper’s main criticism of bag-of-words/cosine is that texts on the same topic may use different vocabulary and thus appear dissimilar. 

Centroid_Terms_as_Text_Represen…


Static embeddings help because semantically related words can still be near each other even without exact overlap.

5) What are the disadvantages?
1. No context sensitivity

This is the biggest limitation.

A static vector gives one embedding per word type, so:

“bank” in finance and “bank” in river bank share one vector

“java” as island vs programming language share one vector

For TRCs, this can produce the wrong representative term.

2. Weak handling of polysemy and sense shifts

If the document is about a less frequent sense of a word, the centroid may be pulled toward the dominant sense in the embedding space.

3. Tendency toward generic terms

Averaging static vectors often moves the representation toward high-frequency semantic hubs, producing vague TRCs like:

system

process

method

thing

This is a classic “centroid drift” problem.

4. Limited abstraction control

The original paper can sometimes surface more abstract terms through the structure of the corpus graph, not just surface similarity. For example, it discusses how a text about “steering wheels” could relate to “car parts” or “car.” 

Centroid_Terms_as_Text_Represen…


Static embeddings alone do not reliably recover that kind of structured abstraction unless the embedding space happens to encode it well, or unless you add lexical resources or a graph.

5. OOV and rare term issues

Especially problematic in technical domains, low-resource languages, or rapidly changing vocabulary.

6. Less robust than contextual embeddings for nuanced semantics

Contextual models can produce document-conditioned term representations, which is usually better for:

disambiguation

multi-topic documents

figurative usage

domain transfer

6) Static TRCs vs contextual/dynamic embeddings

Here is the main tradeoff.

Static embeddings are preferable when:

you want a lightweight, interpretable system

topical semantics matter more than fine-grained sense disambiguation

you have many short texts

computational cost is important

the domain vocabulary is relatively stable

you want a transparent candidate-ranking procedure

Contextual embeddings are preferable when:

words are highly polysemous

documents are nuanced or sentence meaning matters

you need strong semantic matching

the task depends on context-specific interpretation

you want better performance on modern retrieval/classification benchmarks

Best practical view

Static embeddings are not the best general semantic model today, but they are still very reasonable for TRC-style symbolic summarization, especially when the goal is:

give one representative term

cluster topically similar texts

support explainable indexing

A contextual model gives a stronger document representation, but turning it into a clean symbolic representative term is often less direct.

7) Which applications are most suitable?

The paper suggests text clustering, classification, short-text matching, and search-related applications, especially where wording differs across similar texts. It also notes potential usefulness for short texts and search queries. 

Centroid_Terms_as_Text_Represen…

 

Centroid_Terms_as_Text_Represen…

Using static-embedding TRCs, the most suitable real-world applications are:

A. Short-text clustering

Examples:

news headlines

support tickets

forum posts

search queries

social media posts

Why it fits:

short texts are sparse in bag-of-words form

a TRC can stabilize their representation

B. Query expansion and search assistance

A TRC can act as a normalized semantic anchor for a query or document. This matches the paper’s broader motivation around search and semantic matching. 

Centroid_Terms_as_Text_Represen…

 

Centroid_Terms_as_Text_Represen…

C. Topic labeling

After clustering documents, a TRC can provide a human-readable cluster label.

D. Document indexing / browsing

Instead of full-keyword lists, each document can expose one or a few representative terms for navigation.

E. Explainable recommendation

For article or product recommendation, TRCs give a simple semantic explanation:

“recommended because both items map to hybrid vehicle”

F. Domain-specific corpora with controlled vocabulary

Examples:

medical abstracts

legal decisions

product catalogs

patent abstracts

These work especially well if you can train or fine-tune static embeddings on the domain corpus.

8) Where is this approach a poor fit?

It is weaker for:

question answering

NLI / entailment

sentiment with compositional negation

multi-hop reasoning

dialogue understanding

tasks where sentence structure strongly changes meaning

In those settings, a single static term representative is usually too coarse.

9) Recommended implementation variants

If you actually want to build this, I would recommend one of these three variants.

Variant 1: Pure static medoid TRC

Best for a simple baseline.

preprocess document

keep content words

compute weighted average distance from each candidate term to document terms

choose minimizing candidate

Good for:

interpretable prototype

clustering baseline

Variant 2: Static centroid + lexical filtering

Compute document centroid vector, then retrieve nearest candidate terms subject to:

noun-only

frequency thresholds

optional domain lexicon

optional anti-genericity penalty

This often improves output quality.

Variant 3: Hybrid TRC

Use:

corpus co-occurrence graph

static embeddings

weighted combination of graph distance and embedding distance

This is the closest modernized form of the paper’s idea and usually the best design if quality matters.

10) Bottom line

It is absolutely possible to implement TRCs with static word embeddings.

The most principled way is to treat a TRC as a vocabulary term that minimizes average semantic distance to the document’s important words, using static embedding distance instead of graph shortest-path distance. That gives you an embedding-space version of the paper’s centroid-term idea, including the important property that the representative term can be absent from the document itself. 

Centroid_Terms_as_Text_Represen…

But there is an important caveat: static embeddings give you a workable approximation, not a full replacement for the original method’s corpus-level relational structure. The paper’s method depends on a reference co-occurrence graph and on distances induced by that graph, which lets it capture corpus-derived topical relations and abstractions more explicitly than simple vector averaging does. 

Centroid_Terms_as_Text_Represen…

So the practical conclusion is:

Yes, static embeddings can implement TRC-like representatives.

Best simple formulation: medoid-like nearest representative term in embedding space.

Best robust formulation: hybrid co-occurrence graph + static embeddings.

Best use cases: short-text clustering, interpretable retrieval, topic labeling, document indexing, and domain-specific semantic grouping.

# Tasks 2, 3, and 4

## Load data

In [1]:
import sqlite3 
import pandas as pd

sql = sqlite3.connect("wiki_articles_diseases_extended.db")

df = pd.read_sql_query("SELECT * from wiki_articles_diseases_extended", sql)

In [2]:
df

,title,text,name,url,datePublished,dateModified,headline,nouns,adjectives,verbs,lemmas,nav,entities,noun_chunks,no_tokens,no_sentences,no_noun_chunks
0,Acne,"From Wikipedia, the free encyclopedia\n\n\nSki...",Acne,https://en.wikipedia.org/wiki/Acne,2002-08-22T01:27:34Z,2025-01-10T12:50:12Z,sebaceous gland's disease characterized by are...,skin|condition|pimple|article|skin|disease|ado...,common#other|acneiform#medical|also|long|dead#...,characterize|be#see#know|be|occur#include#affe...,|skin|condition|characterize|by|pimple|this|a...,skin|condition|characterize|pimple|article|be|...,ACK-nee#80%#azelaic acid#2015|approximately 63...,Skin condition|pimples|This article|a skin di...,10911.0,481.0,2349.0
1,Alcoholic liver disease,"From Wikipedia, the free encyclopedia\n\n\n\n\...",Alcoholic liver disease,https://en.wikipedia.org/wiki/Alcoholic_liver_...,2003-08-29T06:49:59Z,2024-12-10T17:39:27Z,medical condition,condition|Alcoholic|liver|disease|ALD|alcohol|...,medical|also|alcoholic|chronic#major|western|e...,call|relate|be|encompass|include#be|be|lead#wi...,#medical|condition|Alcoholic|liver|disease|(...,medical|condition|Alcoholic|liver|disease|ALD|...,ARLD#Western#More than 90%|about 25%|15%#Decem...,Medical condition Alcoholic liver disease|ALD|...,2177.0,110.0,461.0
2,Alcoholism,"From Wikipedia, the free encyclopedia\n\n\nPro...",Alcoholism,https://en.wikipedia.org/wiki/Alcoholism,2001-12-20T15:13:43Z,2025-01-19T22:28:08Z,broad term for problems with alcohol,alcohol|consumption|article|alcohol|abuse|heal...,problematic|excessive|chronic|significant#gene...,be|result#see#redirect#see#see#see#be|cause#re...,#problematic|excessive|alcohol|consumption|th...,problematic|excessive|alcohol|consumption|arti...,The World Health Organization|283 million|2016...,Problematic excessive alcohol consumption|This...,9296.0,376.0,2086.0
3,Allergic rhinitis,"From Wikipedia, the free encyclopedia\n\n\nNas...",Allergic rhinitis,https://en.wikipedia.org/wiki/Allergic_rhinitis,2003-12-24T04:16:09Z,2024-11-26T19:41:50Z,human disease డ్యూ తో inflammation in nose oc...,inflammation|allergen|air|Hay|fever#use|Hay|fe...,nasal|here#other#medical|allergic|seasonal|imm...,redirect#see#be|call|be|occur|overreact#be|cla...,|nasal|inflammation|due|to|allergen|in|the|ai...,nasal|inflammation|allergen|air|Hay|fever|redi...,Nasal#Hay#seasonal#minutes#the year#more than ...,"Nasal inflammation|allergens|""Hay fever#other...",4093.0,178.0,867.0
4,Allergy,"From Wikipedia, the free encyclopedia\n\n\nImm...",Allergy,https://en.wikipedia.org/wiki/Allergy,2002-06-07T10:03:08Z,2025-01-21T15:29:18Z,immune system response to a substance that mos...,system|response|substance|people|journal|Aller...,immune|most|well#medical|also|allergic|various...,tolerate|see#see#know|be|cause#include#may|inc...,|immune|system|response|to|a|substance|that|m...,immune|system|response|substance|most|people|t...,about 20%|10%|8%|about 20%#about 1–18%#0.05–2%...,Immune system response|a substance|most peopl...,8224.0,341.0,1841.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228,Waterborne disease,"From Wikipedia, the free encyclopedia\n\n(Redi...",Waterborne disease,https://en.wikipedia.org/wiki/Waterborne_disease,2006-03-18T13:23:48Z,2024-11-30T14:28:25Z,disease caused by pathogenic microorganisms th...,Waterborne|disease#disease|microorganism|water...,pathogenic|medical|adverse|human|such|pathogen...,redirect#cause|transmit|be|mean|cause|be|trans...,#(|redirect|from|Waterborne|disease|)#disease...,redirect|Waterborne|disease#disease|cause|path...,Waterborne#47 #Nematoda#one|Dracunculiasis#Ne...,Waterborne diseases#Diseases|pathogenic microo...,2901.0,112.0,698.0
229,White coat hypertension,"From Wikipedia, the free encyclopedia\n\n\nMed...",White coat hypertension,https://en.wikipedia.org/wiki/White_coat_hyper...,2004-11-21T03:20:18Z,2024-08-28T21:13:13Z,medical condition,condition|coat|hypertension|WHT|coat|syndrome|...,medical|white|also|white|labile|normal|cl

## Vectorize data

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from spacy.lang.en.stop_words import STOP_WORDS as stop_words

tfidf_vectorizer = TfidfVectorizer(stop_words=list(stop_words), min_df=10, sublinear_tf=True, use_idf=True)
tfidf_vectors = tfidf_vectorizer.fit_transform(df["nav"])


c:\Users\aababc\anaconda3\envs\nlp_env\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


## Calculate (or load) word vectors

In [ ]:
#!pip install "gensim>=4.0.0"

In [ ]:
# import regex as re
# from spacy.lang.en.stop_words import STOP_WORDS as stop_words
# gensim_words = [[w for w in re.split(r'[\\|\\#]', doc.lower()) if w not in stop_words]
#                     for doc in df["nav"]]

In [ ]:
# from gensim.models import Word2Vec
# w2v = Word2Vec(gensim_words, min_count=5)
# w2v.wv.save_word2vec_format("wiki_articles_diseases_extended.w2v")

In [4]:
from gensim.models import KeyedVectors
w2v = KeyedVectors.load_word2vec_format("wiki_articles_diseases_extended.w2v")

In [8]:
print(w2v)
print(tfidf_vectors.shape)

KeyedVectors<vector_size=100, 6544 keys>
(233, 4264)


In [19]:
import numpy as np

def get_doc_embedding(doc_idx, tfidf_matrix, tfidf_model, w2v_model):
    # 1. 단어 목록과 해당 문서의 TF-IDF 값들 가져오기
    terms = tfidf_model.get_feature_names_out()
    row = tfidf_matrix[doc_idx].toarray().flatten()
    
    # 2. 0이 아닌 TF-IDF를 가진 단어들만 필터링
    nonzero_indices = row.nonzero()[0]
    
    vec_sum = np.zeros(w2v_model.vector_size)
    weight_sum = 0
    
    for idx in nonzero_indices:
        word = terms[idx]
        weight = row[idx]
        
        # 3. Word2Vec에 해당 단어가 있을 때만 계산
        if word in w2v_model:
            vec_sum += (w2v_model[word] * weight) # 벡터 * 가중치
            weight_sum += weight
            
    # 4. 가중 평균 리턴 (0으로 나누기 방지)
    return vec_sum / weight_sum if weight_sum > 0 else vec_sum

# 233개 문서에 대해 실행하여 리스트 생성
doc_embeddings_list = [get_doc_embedding(i, tfidf_vectors, tfidf_vectorizer, w2v) 
                       for i in range(len(df))]

print(len(doc_embeddings_list))
print(len(doc_embeddings_list[0]))

233
100


In [20]:
# 최종 결과 확인 (과제 제출용 출력)
print(f"1. Total Document Embeddings: {len(doc_embeddings_list)}")
print(f"2. Dimension of each embedding: {len(doc_embeddings_list[0])}")
print(f"3. First Document Embedding (Partial):\n{doc_embeddings_list[0][:10]} ...") 
# 너무 기니까 앞의 10개만 보여주는 센스!

1. Total Document Embeddings: 233
2. Dimension of each embedding: 100
3. First Document Embedding (Partial):
[-0.09114686  0.63718686 -0.13160943 -0.24751765 -0.11235455  0.05026543
 -0.13441707  0.2915141  -0.24759101 -0.25959623] ...


In [22]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. "obesity" 워드 벡터 가져오기
if "obesity" in w2v:
    # 100차원 벡터를 (1, 100) 모양으로 변환
    query_vec = w2v["obesity"].reshape(1, -1)
    
    # 2. 233개 문서 임베딩을 행렬로 변환 (233, 100)
    doc_matrix = np.array(doc_embeddings_list)
    
    # 3. 코사인 유사도 계산
    # (1, 100)과 (233, 100)을 비교하여 233개의 유사도 점수 산출
    similarities = cosine_similarity(query_vec, doc_matrix).flatten()
    
    # 4. 가장 높은 점수의 인덱스 찾기
    best_idx = np.argmax(similarities)
    best_title = df.iloc[best_idx]['title']
    best_score = similarities[best_idx]
    
    print(f"가장 유사한 문서 제목: {best_title}")
    print(f"유사도 점수: {best_score:.4f}")
else:
    print("'obesity' 단어가 모델 사전에 없습니다.")

가장 유사한 문서 제목: Overweight
유사도 점수: 0.7034


$$
Q = \begin{pmatrix} 
1 & 2 & 1 \\ 
2 & 0 & 1 \\ 
1 & 1 & 2 
\end{pmatrix}, \quad 
K = \begin{pmatrix} 
1 & 1 & 0 \\ 
0 & 2 & 1 \\ 
1 & 0 & 2 
\end{pmatrix}
$$

a) (4 points) Compute the raw attention score matrix $$𝑆 = 𝑄𝐾^𝑇$$. Show your calculations for each entry
of the matrix. Note: Do not apply scaling or softmax. Please typeset all matrices and mathematical
expressions using LaTeX in Markdown cells in Jupyter Notebook.

$$ S = \begin{pmatrix} 1& 2& 1 \\ 2& 0& 1 \\ 1& 1& 2 \end{pmatrix} \cdot \begin{pmatrix} 1& 0& 1 \\ 1& 2& 0 \\ 0& 1& 2 \end{pmatrix} $$

$$ S = \begin{pmatrix} 3 & 5 & 3 \\ 2& 1& 4 \\ 2& 4& 5 \end{pmatrix} $$

b) (3 points) Interpret the meaning of the raw attention scores for token T2 (second row of S).
In doing so, indicate which token T2 attends to most strongly and explain what this implies about the
contextualized representation that will be computed for T2 (the actual computation is not required).

The raw attention scores for $T2$ are $[2, 1, 4]$.

Observation: $T2$ attends most strongly to $T3$ (score: 4).

Implication: $T3$ provides the most significant contextual information for $T2$ in this sequence.


Contextualized Representation: The final representation for $T2$ will be a weighted sum heavily influenced by $T3$. This ensures $T2$’s vector incorporates the semantic relationship with $T3$, making it context-dependent rather than static.

a) (4 points) Discuss the main limitations of static word embeddings and how they arise from the
underlying computation process.

The primary limitation is the lack of context-sensitivity (Polysemy problem). Static embeddings like Word2Vec assign a single fixed vector to each word regardless of its usage. This arises because the computation process aggregates global co-occurrence statistics into one representation, failing to distinguish between different meanings of the same word (e.g., "bank" as a financial institution vs. a river bank) in a specific sentence.

b) (4 points) Explain how static word embeddings still play a role in Transformer-based models such
as BERT. In doing so, also discuss how such models address or mitigate the out-of-vocabulary (OOV)
problem compared to classical static word embedding methods.

In models like BERT, static embeddings still serve as the initial input representation (Lookup Table) before any contextual interaction occurs. However, these models mitigate the Out-of-Vocabulary (OOV) problem by using Subword Tokenization (e.g., WordPiece). Unlike classical methods that use whole-word vocabularies, subword models break unknown words into known segments, allowing the model to represent virtually any word and reducing the reliance on a fixed, inflexible vocabulary.